In [ ]:
# Cell 1: Install deps + check GPU
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader
!pip install transformers accelerate bitsandbytes peft datasets sentence-transformers trl pyyaml -q
print("Deps installed")


In [ ]:
# Cell 2: Clone CogMem + load tasks
REPO_BRANCH = "master"
!if [ ! -d /notebooks/CogMem/.git ]; then git clone --branch {REPO_BRANCH} --single-branch https://github.com/tungooxx/CogMem.git /notebooks/CogMem; else cd /notebooks/CogMem && git fetch origin && git checkout {REPO_BRANCH} && git pull --ff-only origin {REPO_BRANCH}; fi
!cd /notebooks/CogMem && pip install -e . --no-deps -q

import sys
import json
from pathlib import Path
from datasets import load_dataset

if "/notebooks/CogMem" not in sys.path:
    sys.path.insert(0, "/notebooks/CogMem")

TASKS_PATH = "/notebooks/bigcodebench_tasks.jsonl"
TASK_LIMIT = 300

if not Path(TASKS_PATH).exists():
    ds = load_dataset("bigcode/bigcodebench", split="v0.1.4")
    tasks = []
    for item in ds:
        tasks.append({
            "task_id": item["task_id"],
            "instruct_prompt": item.get("instruct_prompt", ""),
            "complete_prompt": item.get("complete_prompt", ""),
            "test": item.get("test", ""),
            "canonical_solution": item.get("canonical_solution", ""),
            "entry_point": item.get("entry_point", ""),
            "libs": item.get("libs", []),
        })
    with open(TASKS_PATH, "w") as f:
        for task in tasks:
            f.write(json.dumps(task) + chr(10))
else:
    tasks = []
    with open(TASKS_PATH) as f:
        for line in f:
            if line.strip():
                tasks.append(json.loads(line))

if TASK_LIMIT:
    tasks = tasks[:TASK_LIMIT]

print(Path('/notebooks/CogMem').resolve())
!cd /notebooks/CogMem && git branch --show-current && git rev-parse --short HEAD
print("Tasks loaded:", len(tasks))


In [ ]:
# Cell 3: Prepare split manifest + new-architecture configs
from cogmem.config import CogMemConfig
from cogmem.consolidation.experiment import (
    NewArchitectureExperimentConfig,
    prepare_new_arch_task_split,
    load_new_arch_runtime,
    run_new_arch_episode_collection,
    build_new_arch_skill_cards,
    run_new_arch_qstar_cycle,
)

RESET_COLLECTION_PROGRESS = False
RUN_QSTAR_TRAINING = False

NOTEBOOK_CONFIG = NewArchitectureExperimentConfig(
    experiment_dir="/notebooks/cogmem_new_architecture",
    manifest_path="/notebooks/cogmem_new_architecture/bigcodebench_manifest.json",
    memory_bank_path="/notebooks/cogmem_new_architecture/memory_bank.json",
    skills_path="/notebooks/cogmem_new_architecture/skill_cards.json",
    model_name="Qwen/Qwen2.5-3B-Instruct",
    task_limit=TASK_LIMIT,
    max_attempts=3,
    temperature=0.0,
)

split_result = prepare_new_arch_task_split(tasks, config=NOTEBOOK_CONFIG)
manifest = split_result["manifest"]
train_tasks = split_result["train_tasks"]
dev_tasks = split_result["dev_tasks"]
test_tasks = split_result["test_tasks"]

COGMEM_CONFIG = CogMemConfig(
    project_dir="/notebooks/CogMem",
    bigcodebench_memory_bank=NOTEBOOK_CONFIG.memory_bank_path,
    memory_bank_path=NOTEBOOK_CONFIG.memory_bank_path,
    adapters_dir="/notebooks/cogmem_new_architecture/adapters",
    adapter_registry_path="/notebooks/cogmem_new_architecture/adapters/registry.json",
    experiments_dir="/notebooks/cogmem_new_architecture/experiments",
    logs_dir="/notebooks/cogmem_new_architecture/logs",
    skills_dir="/notebooks/cogmem_new_architecture/skills",
    active_model_hf=NOTEBOOK_CONFIG.model_name,
    base_model=NOTEBOOK_CONFIG.model_name,
    quantization_bits=0,
    use_dora=False,
    generator_rank=8,
    generator_alpha=16,
    verifier_rank=8,
    verifier_alpha=16,
    generator_batch_size=1,
    generator_max_seq_length=1024,
    generator_sft_epochs=1,
    generator_dpo_epochs=1,
    verifier_epochs=1,
    min_dpo_pairs=9999,
    allowed_manifest_ids=[manifest["manifest_id"]],
    require_manifest_match=True,
    bigcodebench_eval_label="bigcodebench_cl",
)

print("Manifest:", manifest["manifest_id"])
print("Train tasks:", len(train_tasks))
print("Dev tasks  :", len(dev_tasks))
print("Test tasks :", len(test_tasks))
print("Memory bank:", NOTEBOOK_CONFIG.memory_bank_path)
print("Skill cards:", NOTEBOOK_CONFIG.skills_path)


In [ ]:
# Cell 4: Load local HF runtime for episode collection
import torch

base_model, tokenizer, llm_client = load_new_arch_runtime(
    model_name=NOTEBOOK_CONFIG.model_name,
)

free = torch.cuda.mem_get_info()[0] / 1024**3
print(f"Runtime loaded. Free VRAM: {free:.1f} GB")


In [ ]:
# Cell 5: Collect typed episodes on the train split
collection_result = run_new_arch_episode_collection(
    train_tasks,
    llm_client,
    config=NOTEBOOK_CONFIG,
    memory_bank_path=NOTEBOOK_CONFIG.memory_bank_path,
    reset_progress=RESET_COLLECTION_PROGRESS,
    verbose=True,
)

print()
print("=" * 60)
print("NEW-ARCH EPISODE COLLECTION COMPLETE")
print("Tasks processed    :", collection_result["tasks_processed"])
print("New episodes       :", collection_result["new_episodes"])
print("Episodes total     :", collection_result["episodes_total"])
print("New successes      :", collection_result["successes_this_run"])
print("Elapsed (min)      :", round(collection_result["elapsed_minutes"], 1))
print("Progress file      :", collection_result["progress_path"])
print("Memory bank path   :", collection_result["memory_bank_path"])
print("Summary metrics    :", collection_result["summary_metrics"])


In [ ]:
# Cell 6: Inspect typed episodic memory bank
from collections import Counter
from cogmem.memory.memory_bank import MemoryBank

bank = MemoryBank.load(NOTEBOOK_CONFIG.memory_bank_path)
metrics = bank.summary_metrics()
print("Episodes:", len(bank))
print("Summary metrics:")
for key, value in metrics.items():
    print(f"  {key}: {value}")

task_type_counts = Counter(ep.get("task_type", "general") for ep in bank)
error_family_counts = Counter(ep.get("error_family") or "None" for ep in bank)
split_counts = Counter(ep.get("split_name") or "unspecified" for ep in bank)

print()
print("Task types:")
for key, value in sorted(task_type_counts.items()):
    print(f"  {key}: {value}")

print()
print("Error families:")
for key, value in error_family_counts.most_common(10):
    print(f"  {key}: {value}")

print()
print("Split counts:")
for key, value in sorted(split_counts.items()):
    print(f"  {key}: {value}")


In [ ]:
# Cell 7: Build + validate procedural skill cards
skill_result = build_new_arch_skill_cards(
    NOTEBOOK_CONFIG.memory_bank_path,
    COGMEM_CONFIG,
    skills_path=NOTEBOOK_CONFIG.skills_path,
)

print("Episodes total     :", skill_result["episodes_total"])
print("Eligible episodes  :", skill_result["eligible_episodes"])
print("Available episodes :", skill_result["available_episodes"])
print("Holdout episodes   :", skill_result["holdout_episodes"])
print("Task type counts   :", skill_result["task_type_counts"])
print("Skill summary      :", skill_result["skill_summary"])
print("Training pairs     :", skill_result["training_pairs"])
print("Preference pairs   :", skill_result["preference_pairs"])

print()
print("Top skill cards:")
for row in skill_result["skill_rows"]:
    print("Skill:", row["skill_id"])
    print(
        "  status:", row["status"],
        "| task_type:", row["task_type"],
        "| domain:", row["domain"],
        "| error_family:", row["error_family"],
    )
    print(
        "  evidence:", row["source_episode_count"],
        "| matched:", row["matched_episodes"],
        "| confidence:", round(row["confidence"], 3),
        "| transfer:", round(row["transfer_gain"], 3),
        "| negative_transfer:", round(row["negative_transfer_rate"], 3),
        "| success_rate:", round(row["success_rate"], 3),
    )
    print("  triggers:", row["triggers"])
    print("  plan_steps:", row["plan_steps"])
    print("  anti_patterns:", row["anti_patterns"])


In [ ]:
# Cell 8: Optional Q-STaR consolidation cycle (adapter training)
if RUN_QSTAR_TRAINING:
    import gc
    import torch

    base_model = None
    tokenizer = None
    llm_client = None
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    qstar_result = run_new_arch_qstar_cycle(
        NOTEBOOK_CONFIG.memory_bank_path,
        COGMEM_CONFIG,
        cycle=0,
    )
    print(json.dumps(qstar_result, indent=2, ensure_ascii=False))
else:
    print("RUN_QSTAR_TRAINING is False. Set it to True to train generator/verifier adapters.")
    print("Promoted skills available:", len(skill_result["promoted_skill_ids"]))
    print("Skill cards path:", skill_result["skills_path"])
